# Private SEOCHO + Opik workflow — your identity, your traces

A personalized end-to-end notebook that ties **your** identity (USER_ID, AGENT_ID, SESSION_ID, WORKSPACE_ID, plus arbitrary metadata) to every operation across SEOCHO's four design surfaces:

1. **Ontology design** — load FIBO-style TTL files, **plus** them together, **minus** restricted classes, save back to TTL
2. **LLM backend** — direct calls with explicit Opik spans tagged with your identity
3. **Agent tool_use design** — register a custom tool, run it, observe the trace
4. **Pattern design** — pick an `AgentConfig` execution mode + routing policy, run end-to-end against an embedded graph store

**Tracing.** Every cell that does interesting work emits an Opik span via `seocho.tracing.log_span(...)` with `metadata={user_id, agent_id, session_id, workspace_id, **user_meta}`. If Opik isn't running, the notebook falls back to a JSONL trace file under `./.seocho/private_<user>/` so the workflow still produces durable traces.

**Privacy.** Anything written by this notebook lives under `./.seocho/private_<user>/` which is in the workdir's `.gitignore` already — the artifacts don't leak into version control. The notebook itself is a reusable template; commit a copy *under your fork* if you want, or run it as-is.

**Prereqs.** `OPENAI_API_KEY` in `.env`. Optionally `make opik-up` from the main compose to bring up Opik, otherwise this notebook auto-falls-back to JSONL traces. The `examples/Dockerfile.tutorials` image has all Python deps.

## 1. Identity + tracing setup

In [ ]:
import json
import os
import sys
import time
import uuid
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")

# === EDIT ME — your identity ===========================================
USER_ID      = os.environ.get("SEOCHO_USER_ID")      or "alice"
AGENT_ID     = os.environ.get("SEOCHO_AGENT_ID")     or f"{USER_ID}-agent"
SESSION_ID   = os.environ.get("SEOCHO_SESSION_ID")   or uuid.uuid4().hex[:12]
WORKSPACE_ID = os.environ.get("SEOCHO_WORKSPACE_ID") or f"private-{USER_ID}"
USER_META    = json.loads(
    os.environ.get("SEOCHO_USER_META", '{"team":"research","project":"private-kg","env":"local"}')
)
# ========================================================================

PRIVATE_DIR = ROOT / ".seocho" / f"private_{USER_ID}"
PRIVATE_DIR.mkdir(parents=True, exist_ok=True)

IDENTITY = {
    "user_id": USER_ID,
    "agent_id": AGENT_ID,
    "session_id": SESSION_ID,
    "workspace_id": WORKSPACE_ID,
    "llm_provider": LLM_PROVIDER,
    "llm_model": LLM_MODEL,
    **USER_META,
}
print(json.dumps(IDENTITY, indent=2))
print(f"\nArtifacts will be written under: {PRIVATE_DIR}")

In [ ]:
from seocho.tracing import (
    current_backend_names,
    enable_tracing,
    flush_tracing,
    is_backend_enabled,
    is_observability_degraded,
    log_span,
)

# Try Opik first; fall back to JSONL if unreachable.
opik_url = os.environ.get("OPIK_URL_OVERRIDE") or os.environ.get("OPIK_URL")
opik_workspace = os.environ.get("OPIK_WORKSPACE", "default")
opik_project = os.environ.get("OPIK_PROJECT_NAME", f"seocho-private-{USER_ID}")
opik_mode = os.environ.get("SEOCHO_TRACE_OPIK_MODE", "self_host")

ok = False
if opik_url:
    ok = enable_tracing(
        backend=["opik", "jsonl"],
        url=opik_url,
        workspace=opik_workspace,
        project_name=opik_project,
        opik_mode=opik_mode,
        output=str(PRIVATE_DIR / "traces.jsonl"),
    )
if not ok:
    ok = enable_tracing(
        backend="jsonl",
        output=str(PRIVATE_DIR / "traces.jsonl"),
    )

print("Active backends:", current_backend_names())
print("Opik enabled:", is_backend_enabled("opik"))
print("Observability degraded:", is_observability_degraded())

def traced(name: str, *, input_data=None, output_data=None, tags=None):
    """Convenience: emit a span tagged with the active identity."""
    log_span(
        name,
        input_data=input_data or {},
        output_data=output_data or {},
        metadata=dict(IDENTITY),
        tags=list(tags or []) + [f"user:{USER_ID}", f"session:{SESSION_ID}"],
    )

traced("private.session.start", input_data={"identity": IDENTITY}, tags=["bootstrap"])

## 2. Ontology design — TTL plus / minus

SEOCHO's stock I/O is JSON-LD primary plus YAML; `Ontology.merge` is union-shaped. The `examples/ontology_io` helper fills two adjacent gaps:

- **TTL load/save** so you can compose from `.ttl` files (FIBO is published as TTL).
- **`+` and `-` operators** so a multi-source ontology composition reads like algebra.

Mapping: `owl:Class` → `NodeDef`, `owl:ObjectProperty` (with `rdfs:domain`/`rdfs:range`) → `RelDef`, `owl:DatatypeProperty` → property on its domain class, `skos:altLabel` → aliases.

In [ ]:
from seocho import Ontology

TTL_DIR = ROOT / "examples" / "datasets" / "ttl"

with_op_t0 = time.perf_counter()
base       = Ontology.from_ttl(TTL_DIR / "private_base.ttl",       name="private_base")
extension  = Ontology.from_ttl(TTL_DIR / "private_extension.ttl",  name="private_extension")
restricted = Ontology.from_ttl(TTL_DIR / "private_restricted.ttl", name="private_restricted")

# Operator overloads ship in seocho core: + is merge (union), - is subtract.
composed = base + extension - restricted
composed.name = f"{USER_ID}/private_kg"
composed.package_id = composed.name
composed.namespace = "https://seocho.dev/private/"
elapsed_ms = (time.perf_counter() - with_op_t0) * 1000

print("Base       — labels:", sorted(base.nodes.keys()))
print("Extension  — labels:", sorted(extension.nodes.keys()))
print("Restricted — labels:", sorted(restricted.nodes.keys()))
print("Composed   — labels:", sorted(composed.nodes.keys()))
print("Composed   — rels  :", sorted(composed.relationships.keys()))

# Persist the composed ontology in three formats
composed_ttl    = composed.to_ttl(PRIVATE_DIR / "composed.ttl")
composed_jsonld = composed.to_jsonld(PRIVATE_DIR / "composed.jsonld")
composed.to_yaml(PRIVATE_DIR / "composed.yaml")
shacl_payload   = composed.to_shacl()
(PRIVATE_DIR / "composed.shacl.json").write_text(json.dumps(shacl_payload, indent=2))

traced(
    "private.ontology.compose",
    input_data={
        "sources": {
            "base": list(base.nodes.keys()),
            "extension": list(extension.nodes.keys()),
            "restricted": list(restricted.nodes.keys()),
        }
    },
    output_data={
        "composed_labels": list(composed.nodes.keys()),
        "composed_rels": list(composed.relationships.keys()),
        "saved_ttl": str(composed_ttl),
        "elapsed_ms": round(elapsed_ms, 1),
    },
    tags=["ontology", "ttl", "plus-minus"],
)

without_minus = base + extension
removed_labels = sorted(set(without_minus.nodes.keys()) - set(composed.nodes.keys()))
removed_rels = sorted(set(without_minus.relationships.keys()) - set(composed.relationships.keys()))
print(f"\n`- restricted` removed labels: {removed_labels}")
print(f"`- restricted` removed rels:   {removed_rels}")


## 3. LLM backend with attribution

Direct LLM call — every invocation wrapped in a span tagged with your identity. The LLM itself is provider-agnostic via `create_llm_backend`; the trace metadata is what makes the call attributable to *you* in Opik.

In [ ]:
from seocho.store.llm import create_llm_backend

llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

ANSWER_SYSTEM = "You are a concise assistant. Answer in one sentence unless the user asks for more."

def attributed_complete(user_msg: str, *, op: str = "private.llm.complete", system: str = ANSWER_SYSTEM) -> str:
    t0 = time.perf_counter()
    response = llm.complete(system=system, user=user_msg)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    answer = response.text
    log_span(
        op,
        input_data={"user_chars": len(user_msg), "user_preview": user_msg[:120]},
        output_data={"answer_chars": len(answer), "answer_preview": answer[:200], "elapsed_ms": round(elapsed_ms, 1)},
        metadata={**IDENTITY, "model": LLM_MODEL, "provider": LLM_PROVIDER},
        tags=["llm", f"user:{USER_ID}"],
    )
    return answer

demo_q = "In one sentence, why does an ontology improve LLM extraction quality?"
demo_a = attributed_complete(demo_q)
print("Q:", demo_q)
print("A:", demo_a)

## 4. Agent tool_use design

Register a custom tool with seocho's `BaseAgent` / `ToolRegistry`. The tool reads `IDENTITY` at invocation time and stamps it on the span, so any tool call is fully attributable.

(For the full openai-agents SDK runtime, swap this BaseAgent for `AgentsRuntimeAdapter`. We keep this cell minimal so the trace shape is obvious.)

In [ ]:
from extraction.agent_base.base import BaseAgent, register_tool, ToolRegistry

# Tool: report on the composed ontology — schema-aware, identity-tagged.
@register_tool("my_ontology_report")
def my_ontology_report(query: str = "") -> str:
    """Return a brief report on the user's composed ontology."""
    t0 = time.perf_counter()
    summary = {
        "name": composed.name,
        "labels": sorted(composed.nodes.keys()),
        "relationships": sorted(composed.relationships.keys()),
        "version": composed.version,
    }
    output = json.dumps(summary, indent=2)
    log_span(
        "private.tool.my_ontology_report",
        input_data={"query": query},
        output_data={"summary": summary, "elapsed_ms": round((time.perf_counter() - t0) * 1000, 1)},
        metadata={**IDENTITY, "tool": "my_ontology_report"},
        tags=["tool", f"user:{USER_ID}"],
    )
    return output

# Build a minimal agent that exposes the tool.
class PrivateOntologyAgent(BaseAgent):
    def __init__(self):
        super().__init__(
            name=f"private-ontology-agent::{USER_ID}",
            instructions=(
                f"You answer ontology-shape questions for user '{USER_ID}'. "
                "When the user asks about labels, relationships, or schema, "
                "call my_ontology_report and quote its output."
            ),
            tools=[my_ontology_report],
            model="gpt-4o-mini",
        )
    def validate_input(self, input_data) -> bool:
        return True

agent = PrivateOntologyAgent()
print("Registered tools in ToolRegistry:", ToolRegistry.list_tools())
print("\nDirect invoke (without LLM, just to see the trace):")
print(my_ontology_report("shape"))

## 5. Pattern design

`AgentConfig` is the seam where pattern choice (pipeline vs agent vs supervisor), routing policy (latency vs token-efficiency vs information-quality), and per-stage strategies all converge. Pick once; the rest of seocho honors it across `add()` and `ask()`.

All three `AgentConfig` execution modes are valid here:
- `pipeline` — deterministic, no LLM reasoning; lowest cost
- `agent` — LLM-driven tool use per stage
- `supervisor` — top-level supervisor routes between indexing and query (needs `handoff=True`)

We pick `agent` mode with a `balanced()` routing policy and tag the span accordingly.

In [ ]:
from seocho.agent_config import AgentConfig, RoutingPolicy

agent_config = AgentConfig(
    execution_mode="agent",
    routing_policy=RoutingPolicy.balanced(),
)
hints = agent_config.routing_policy.to_agent_hints() if hasattr(agent_config.routing_policy, "to_agent_hints") else {}

traced(
    "private.pattern.choose",
    input_data={
        "execution_mode": agent_config.execution_mode,
        "routing_policy": getattr(agent_config.routing_policy, "name", str(agent_config.routing_policy)),
    },
    output_data={"hints": hints},
    tags=["pattern", "agent-config"],
)
print("Execution mode:  ", agent_config.execution_mode)
print("Routing policy:  ", agent_config.routing_policy)
print("Derived hints:   ", hints)

## 6. End-to-end Seocho `add()` + `ask()` with full identity propagation

We wire the composed ontology and chosen pattern into a `Seocho` client (with embedded `LanceGraphStore` from Tutorial 1 — no external server). Every `add()` and `ask()` is wrapped in a span tagged with your identity.

In [ ]:
from seocho import Seocho
from examples.lance_graph_store import LanceGraphStore

graph_store = LanceGraphStore(uri=str(PRIVATE_DIR / "graph.lance"))
client = Seocho(
    ontology=composed,
    graph_store=graph_store,
    llm=llm,
    workspace_id=WORKSPACE_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    session_id=SESSION_ID,
    agent_config=agent_config,
)

DOCS = [
    "Apple Inc. (AAPL) is headquartered in Cupertino, California. The company was founded by Steve Jobs and reported $383.3 billion in revenue for fiscal year 2023.",
    "Microsoft Corporation employs Satya Nadella as Chief Executive Officer. The company holds a portfolio of patents in cloud computing.",
]

for i, doc in enumerate(DOCS):
    t0 = time.perf_counter()
    memory = client.add(doc, metadata={"source": f"doc_{i}", **USER_META})
    elapsed_ms = (time.perf_counter() - t0) * 1000
    traced(
        "private.add",
        input_data={"doc_index": i, "chars": len(doc)},
        output_data={
            "memory_id": getattr(memory, "memory_id", None),
            "nodes_created": memory.metadata.get("nodes_created") if memory else None,
            "relationships_created": memory.metadata.get("relationships_created") if memory else None,
            "elapsed_ms": round(elapsed_ms, 1),
        },
        tags=["add"],
    )

QUESTIONS = [
    "Where is Apple Inc. headquartered?",
    "Who is Microsoft's CEO?",
]
answers = []
for q in QUESTIONS:
    t0 = time.perf_counter()
    try:
        a = client.ask(q)
    except Exception as exc:
        a = f"<error: {type(exc).__name__}>"
    elapsed_ms = (time.perf_counter() - t0) * 1000
    answers.append({"q": q, "a": a, "elapsed_ms": round(elapsed_ms, 1)})
    traced(
        "private.ask",
        input_data={"question": q},
        output_data={"answer": a, "elapsed_ms": round(elapsed_ms, 1)},
        tags=["ask"],
    )

for row in answers:
    print(f"Q: {row['q']}\nA: {row['a']}\n  ({row['elapsed_ms']} ms)\n")

## 7. Verify in Opik (or read the JSONL)

In [ ]:
traced("private.session.end", input_data={}, output_data={"answers": len(answers)}, tags=["shutdown"])
flush_tracing()

if is_backend_enabled("opik"):
    print(f"✅ Opik active — open: {opik_url or 'http://localhost:5173'}")
    print(f"   workspace: {opik_workspace}")
    print(f"   project:   {opik_project}")
    print(f"   filter on tag user:{USER_ID} or session:{SESSION_ID} to scope to this run.")
else:
    jsonl_path = PRIVATE_DIR / "traces.jsonl"
    print(f"⚠️  Opik not enabled — traces written to: {jsonl_path}")
    print("   Tail the file to inspect spans:")
    print(f"     tail -n 20 {jsonl_path}")
    if jsonl_path.exists():
        lines = jsonl_path.read_text().strip().splitlines()
        print(f"\n{len(lines)} spans recorded. Last 3:")
        for line in lines[-3:]:
            try:
                span = json.loads(line)
                print(f"  - {span.get('name','?')}  user_id={span.get('metadata',{}).get('user_id','?')}")
            except Exception:
                continue

print("\nArtifacts written this session:")
for f in sorted(PRIVATE_DIR.glob("*")):
    print(f"  {f.relative_to(ROOT)}")

## What you can take from here

- **Identity is a metadata pattern, not a feature** — every operation here is just a normal seocho call wrapped in `log_span(metadata=IDENTITY)`. Reuse the `traced(...)` helper at the top of any future notebook.
- **TTL plus/minus is two operators** — `from examples.ontology_io import *` gives you `+` and `-` on `Ontology`; everything else is fixture wiring.
- **Patterns flow through `AgentConfig`** — change `execution_mode` to `"pipeline"` for deterministic runs, `"supervisor"` for routing, and the rest of the notebook keeps working.
- **Opik is opt-in** — bring up the main Opik stack (`make opik-up`) to get a real dashboard; the JSONL fallback gives you the same shape of trace records on disk.

Cleanup: `rm -rf .seocho/private_<your-user>` wipes everything this notebook produced.